In [ ]:
pip install transformers torch pandas indic-transliteration

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from dataclasses import dataclass
from datasets import Dataset
from ai4bharat_transliteration import XlitEngine
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
import evaluate
import logging

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


In [ ]:
@dataclass
class ModelConfig:
    model_name: str = "google/muril-base-cased"
    max_length: int = 100
    batch_size: int = 32
    learning_rate: float = 2e-5
    epochs: int = 5
    num_labels: int = 3  # Neg / Neu / Pos
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

config = ModelConfig()


In [ ]:
class TextPreprocessor:
    def __init__(self):
        self.engine = XlitEngine("ne", beam_width=4, rescore=True)

    def transliterate_text(self, text: str) -> str:
        try:
            result = self.engine.translit_sentence(text)
            return result["ne"]
        except Exception:
            return text


In [ ]:
def load_and_prepare_data(preprocessor, file_path="data.csv"):
    df = pd.read_csv(file_path)

    assert "sentence" in df.columns, "Missing 'sentence' column"
    assert "label" in df.columns, "Missing 'label' column"

    df = df.dropna(subset=["sentence", "label"])
    df["label"] = df["label"].astype(int)

    logger.info("Applying Roman → Devanagari transliteration...")
    df["processed_text"] = df["sentence"].apply(
        preprocessor.transliterate_text
    )

    return Dataset.from_pandas(df)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(config.model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["processed_text"],
        truncation=True,
        max_length=config.max_length
    )


In [ ]:
preprocessor = TextPreprocessor()

dataset = load_and_prepare_data(
    preprocessor,
    file_path="data.csv"   # 👈 your dataset
)

dataset = dataset.train_test_split(test_size=0.2)

tokenized = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["sentence", "processed_text"]
)


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    config.model_name,
    num_labels=config.num_labels
).to(config.device)


In [ ]:
f1_metric = evaluate.load("f1")
acc_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1": f1_metric.compute(
            predictions=preds,
            references=labels,
            average="weighted"
        )["f1"],
        "accuracy": acc_metric.compute(
            predictions=preds,
            references=labels
        )["accuracy"]
    }


In [ ]:
training_args = TrainingArguments(
    output_dir="./results_nepali_sentiment",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=config.learning_rate,
    per_device_train_batch_size=config.batch_size,
    per_device_eval_batch_size=config.batch_size,
    num_train_epochs=config.epochs,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()


In [ ]:
trainer.save_model("./final_nepali_sentiment_model")
print("✅ Training complete. Model saved.")
